# Notebook 16: Final Consolidated Investment Agent System
## Orchestrator + Agents 1, 2, 3 + `confirm_portfolio` + Persistent Investor Profile

This is the **definitive, production-ready** version of the three-agent investment
system. It incorporates all improvements from Notebooks 11–15, plus the two new
features requested:

### New in This Notebook
1. **`confirm_portfolio` tool** — after Agent 1 designs a portfolio, the user
   must explicitly approve or request changes before it is saved to disk.
   Approval triggers Agent 2 to consolidate the new portfolio with existing
   holdings automatically.
2. **Persistent Investor Profile** — a structured `INVESTOR_PROFILE` object
   accumulates across turns. Every time the user provides profile information
   (age, risk tolerance, time horizon, goals, preferred instruments), it is
   extracted and merged into the profile. Agent 1 always receives the
   full enriched profile, so later queries build on earlier context.

### Full Architecture
```
┌───────────────────────────────────────────────────────────────────────┐
│                    ORCHESTRATOR (Master Agent)                         │
│          LLM with 5 tools — routes every query precisely               │
│                                                                        │
│  Tools:                                                                │
│  ① invoke_portfolio_designer  ← build / update portfolio               │
│  ② confirm_portfolio          ← approve draft → save + consolidate     │
│  ③ invoke_portfolio_analyst   ← analytics, volatility, benchmark       │
│  ④ invoke_investment_educator ← explain concepts, personalised         │
│  ⑤ search_web                 ← current prices, market data           │
└───────────────────────────────────────────────────────────────────────┘
```

### Investor Profile Flow
```
Turn 1: "I am 45"                         → profile.age = 45
Turn 2: "moderate risk"                   → profile.risk = moderate
Turn 3: "retiring in 15 years"            → profile.horizon = 15yr
Turn 4: "prefer ETFs, no real estate"     → profile.preferences updated
Turn 5: "build my portfolio"              → Agent 1 receives FULL profile
                                             → Draft portfolio presented
Turn 6: "looks good, save it"             → confirm_portfolio called
                                             → Saved to JSON + Excel
                                             → Agent 2 consolidates
```

### Portfolio Confirmation Flow
```
Agent 1 designs portfolio  →  PENDING state
       │
   User reviews draft
       │
   ┌───┴────────────────┐
   │ 'yes/save/approve' │  → confirm_portfolio(approved=True)
   │                    │    → saves JSON + Excel + triggers Agent 2
   │ 'change X to Y'    │  → invoke_portfolio_designer (refinement turn)
   │ 'cancel/no'        │  → confirm_portfolio(approved=False) → discarded
   └────────────────────┘
```

## How to Use This Notebook

### Prerequisites
Create `../.env`:
```
LLM_PROVIDER=openai
LLM_MODEL=gpt-4o-mini
OPENAI_API_KEY=your-key
SERPER_API_KEY=your-serper-key
```

### Running
Use `Kernel > Restart & Run All`. The automated demo runs all 5 test
queries plus a full portfolio design → confirm → save → consolidate flow.
The interactive cell at the bottom lets you continue the conversation.

### Key Features
| Feature | How to trigger |
|---------|---------------|
| Build portfolio | `"Help me build a portfolio"` or `"I am 45, moderate risk..."` |
| Approve & save | `"yes save it"` / `"looks good"` / `"approve"` |
| Reject / change | `"change the bond allocation to 20%"` / `"cancel"` |
| Analytics | `"How volatile is my portfolio?"` |
| Education | `"What is beta?"` / `"Explain DCA"` |
| Web search | `"What is VTI's current price?"` |

## 1. Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph \
             python-dotenv google-search-results \
             pandas openpyxl pypdf pillow --quiet
print("✅ Packages installed")

## 2. Imports & Configuration

In [ ]:
import json
import os
import time
import copy
from pathlib import Path
from typing import Literal
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

load_dotenv()
load_dotenv('../.env')

try:
    from ai_course_utils import load_llm_from_env, display_config
    USE_COURSE_UTILS = True
    display_config()
except ImportError:
    USE_COURSE_UTILS = False
    def load_llm_from_env(temperature: float = 0.0):
        model = os.getenv('LLM_MODEL', 'gpt-4o-mini')
        return ChatOpenAI(model=model, temperature=temperature)
    print(f"LLM_MODEL : {os.getenv('LLM_MODEL', 'gpt-4o-mini')}")
    print(f"OpenAI key: {'Set ✅' if os.getenv('OPENAI_API_KEY') else 'NOT SET ⚠️'}")
    print(f"Serper key: {'Set ✅' if os.getenv('SERPER_API_KEY') else 'NOT SET ⚠️'}")

print("\nImports successful")

## 3. File Paths

In [ ]:
USER_ID            = "user1"
OUTPUT_DIR         = Path(f"../data/outputs/{USER_ID}")
AGENT1_JSON        = Path("../data/outputs/portfolio.json")
AGENT1_EXCEL       = Path("../data/outputs/portfolio.xlsx")
CONSOLIDATED_JSON  = OUTPUT_DIR / "consolidated_portfolio.json"
CONSOLIDATED_EXCEL = OUTPUT_DIR / "consolidated_portfolio.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path("../data/outputs").mkdir(parents=True, exist_ok=True)

print(f"Agent 1 output : {AGENT1_JSON}")
print(f"Consolidated   : {CONSOLIDATED_JSON}")

## 4. Pre-Loaded Portfolio Data (5 PDFs)

In [ ]:
SAMPLE_PORTFOLIOS = [
    # Portfolio 1 — Conservative Traditional IRA
    {"ticker":"BND",  "company_name":"Vanguard Total Bond Market ETF",           "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":620.45, "share_price":75.20,   "amount_usd":46679,  "gain_loss_usd": 1420.45, "gain_loss_pct": 3.13,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"LQD",  "company_name":"iShares iBoxx Investment Grade Corp Bond",  "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":330.12, "share_price":109.25,  "amount_usd":36049,  "gain_loss_usd":  -920.12, "gain_loss_pct":-2.49,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"SCHP", "company_name":"Schwab US TIPS ETF",                        "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":480.00, "share_price":56.14,   "amount_usd":26947,  "gain_loss_usd":   347.00, "gain_loss_pct": 1.31,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",           "investment_type":"ETF",        "asset_class":"US Equity",    "shares":110.00, "share_price":335.40,  "amount_usd":36894,  "gain_loss_usd":  4102.00, "gain_loss_pct":12.50,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":290.00, "share_price":62.45,   "amount_usd":18110,  "gain_loss_usd":   980.12, "gain_loss_pct": 5.72,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",        "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":200.00, "share_price":53.90,   "amount_usd":10780,  "gain_loss_usd":  -510.34, "gain_loss_pct":-4.52,  "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    # Portfolio 2 — Taxable Stock Heavy
    {"ticker":"AAPL", "company_name":"Apple Inc.",                                "investment_type":"Stock",      "asset_class":"US Equity",    "shares":320.51, "share_price":262.11,  "amount_usd":84022,  "gain_loss_usd":14901.00, "gain_loss_pct":21.54,  "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"MSFT", "company_name":"Microsoft Corp.",                           "investment_type":"Stock",      "asset_class":"US Equity",    "shares":190.00, "share_price":412.34,  "amount_usd":78344,  "gain_loss_usd":20120.00, "gain_loss_pct":34.54,  "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"NVDA", "company_name":"NVIDIA Corporation",                        "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 75.88, "share_price":1038.22, "amount_usd":78716,  "gain_loss_usd":41822.00, "gain_loss_pct":113.70, "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"SPY",  "company_name":"SPDR S&P 500 ETF Trust",                    "investment_type":"ETF",        "asset_class":"US Equity",    "shares": 35.00, "share_price":528.81,  "amount_usd":18508,  "gain_loss_usd": 1840.00, "gain_loss_pct":11.04,  "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"VNQ",  "company_name":"Vanguard Real Estate ETF",                  "investment_type":"ETF",        "asset_class":"Real Estate",  "shares": 40.00, "share_price":88.41,   "amount_usd": 3536,  "gain_loss_usd":  -93.21, "gain_loss_pct":-2.55,  "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    # Portfolio 3 — 401(k)
    {"ticker":"VTTHX","company_name":"Vanguard Target Retirement 2035",           "investment_type":"Mutual Fund","asset_class":"US Equity",    "shares":950.00, "share_price":41.77,   "amount_usd":39681,  "gain_loss_usd": 2211.00, "gain_loss_pct": 5.90,  "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",           "investment_type":"ETF",        "asset_class":"US Equity",    "shares":355.00, "share_price":335.40,  "amount_usd":118767, "gain_loss_usd":10220.00, "gain_loss_pct": 9.42,  "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"BND",  "company_name":"Vanguard Total Bond Market ETF",           "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":830.00, "share_price":75.20,   "amount_usd":62416,  "gain_loss_usd":  -850.00, "gain_loss_pct":-1.35,  "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"IJR",  "company_name":"iShares Core S&P Small-Cap ETF",            "investment_type":"ETF",        "asset_class":"US Equity",    "shares":510.00, "share_price":119.12,  "amount_usd":60742,  "gain_loss_usd": 3545.00, "gain_loss_pct": 6.20,  "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":395.00, "share_price":62.45,   "amount_usd":24650,  "gain_loss_usd": 2032.00, "gain_loss_pct": 9.00,  "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    # Portfolio 4 — Roth IRA Tech
    {"ticker":"TSLA", "company_name":"Tesla Inc.",                                "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 80.00, "share_price":415.11,  "amount_usd":33208,  "gain_loss_usd": -1350.00,"gain_loss_pct":-3.90,  "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"NVDA", "company_name":"NVIDIA Corporation",                        "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 42.00, "share_price":1038.22, "amount_usd":43605,  "gain_loss_usd":22205.00, "gain_loss_pct":103.70, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"MSFT", "company_name":"Microsoft Corp.",                           "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 70.00, "share_price":412.34,  "amount_usd":28863,  "gain_loss_usd": 6110.00, "gain_loss_pct":26.80,  "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"GOOG", "company_name":"Alphabet Inc. Class C",                     "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 50.00, "share_price":165.88,  "amount_usd": 8294,  "gain_loss_usd":  964.00, "gain_loss_pct":13.10,  "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"SPY",  "company_name":"SPDR S&P 500 ETF Trust",                    "investment_type":"ETF",        "asset_class":"US Equity",    "shares": 18.00, "share_price":528.81,  "amount_usd": 9518,  "gain_loss_usd":  820.00, "gain_loss_pct": 9.50,  "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",        "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares": 35.00, "share_price":53.90,   "amount_usd": 1886,  "gain_loss_usd":  -123.00,"gain_loss_pct":-6.50,  "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    # Portfolio 5 — Simple ETF IRA
    {"ticker":"AGG",  "company_name":"iShares Core US Aggregate Bond ETF",        "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":220.00, "share_price":97.51,   "amount_usd":21452,  "gain_loss_usd":  -620.00,"gain_loss_pct":-2.81,  "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":240.00, "share_price":62.47,   "amount_usd":14992,  "gain_loss_usd":   832.00,"gain_loss_pct": 5.88,  "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",        "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":310.00, "share_price":53.90,   "amount_usd":16709,  "gain_loss_usd":  -143.00,"gain_loss_pct":-0.85,  "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"SCHH", "company_name":"Schwab US REIT ETF",                        "investment_type":"ETF",        "asset_class":"Real Estate",  "shares":395.00, "share_price":20.16,   "amount_usd": 7962,  "gain_loss_usd":  -215.00,"gain_loss_pct":-2.63,  "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",           "investment_type":"ETF",        "asset_class":"US Equity",    "shares":145.00, "share_price":335.40,  "amount_usd":48633,  "gain_loss_usd":  5590.00,"gain_loss_pct":13.00,  "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
]

ACCOUNT_METADATA = [
    {"source_file":"Portfolio1.pdf","account_type":"Traditional IRA",  "account_value":182450.22,"cash":9112.44},
    {"source_file":"Portfolio2.pdf","account_type":"Individual/Taxable","account_value":267915.88,"cash":3140.22},
    {"source_file":"Portfolio3.pdf","account_type":"401(k)",            "account_value":342881.55,"cash":2995.00},
    {"source_file":"Portfolio4.pdf","account_type":"Roth IRA",          "account_value":128940.15,"cash":1100.00},
    {"source_file":"Portfolio5.pdf","account_type":"Traditional IRA",  "account_value":104645.50,"cash": 895.44},
]
TOTAL_AUM = sum(a["account_value"] for a in ACCOUNT_METADATA)
print(f"✅ {len(SAMPLE_PORTFOLIOS)} raw holdings | {len(ACCOUNT_METADATA)} accounts | AUM: ${TOTAL_AUM:,.0f}")

## 5. Analytics Engine

In [ ]:
def consolidate_holdings(raw: list[dict]) -> list[dict]:
    """Merge same-ticker holdings across accounts; normalise allocations to 100%."""
    merged: dict[str, dict] = {}
    for h in raw:
        t = (h.get("ticker") or "UNKNOWN").upper().strip()
        if t not in merged:
            merged[t] = {"ticker":t,"company_name":h.get("company_name"),
                         "investment_type":h.get("investment_type"),"asset_class":h.get("asset_class"),
                         "amount_usd":0.0,"shares":0.0,"gain_loss_usd":0.0,"gl_list":[],
                         "source_files":[],"source_accounts":[]}
        r = merged[t]
        for k in ("company_name","investment_type","asset_class"):
            if not r[k] and h.get(k): r[k] = h[k]
        r["amount_usd"]   += float(h.get("amount_usd")    or 0)
        r["shares"]        += float(h.get("shares")        or 0)
        r["gain_loss_usd"] += float(h.get("gain_loss_usd") or 0)
        if h.get("gain_loss_pct") is not None:
            r["gl_list"].append(float(h["gain_loss_pct"]))
        for fk, lst in (("source_file", r["source_files"]), ("source_account", r["source_accounts"])):
            v = h.get(fk, "")
            if v and v not in lst: lst.append(v)
    lst = list(merged.values())
    total = sum(h["amount_usd"] for h in lst)
    for h in lst:
        h["allocation_pct"] = round(h["amount_usd"]/total*100, 4) if total else 0
        h["gain_loss_pct"]  = round(sum(h["gl_list"])/len(h["gl_list"]), 2) if h["gl_list"] else None
        del h["gl_list"]
    diff = round(100.0 - sum(h["allocation_pct"] for h in lst), 4)
    if diff and lst:
        max(lst, key=lambda x: x["allocation_pct"])["allocation_pct"] = round(
            max(lst, key=lambda x: x["allocation_pct"])["allocation_pct"] + diff, 4)
    lst.sort(key=lambda x: -x["allocation_pct"])
    return lst


def compute_analytics(holdings: list[dict]) -> dict:
    """Compute full portfolio analytics from a holdings list."""
    total_usd  = sum(h["amount_usd"]    for h in holdings)
    total_gain = sum(h["gain_loss_usd"] for h in holdings)
    total_cost = total_usd - total_gain
    total_ret  = (total_gain / total_cost * 100) if total_cost > 0 else 0
    gl_pcts    = [h["gain_loss_pct"] for h in holdings if h.get("gain_loss_pct") is not None]
    mean_gl    = sum(gl_pcts)/len(gl_pcts) if gl_pcts else 0
    variance   = sum((x-mean_gl)**2 for x in gl_pcts)/(len(gl_pcts)-1) if len(gl_pcts) > 1 else 0
    vol        = variance ** 0.5
    class_map: dict[str, dict] = {}
    for h in holdings:
        ac = h.get("asset_class", "Other")
        if ac not in class_map: class_map[ac] = {"allocation_pct":0, "amount_usd":0, "gain_loss_usd":0}
        class_map[ac]["allocation_pct"] += h["allocation_pct"]
        class_map[ac]["amount_usd"]     += h["amount_usd"]
        class_map[ac]["gain_loss_usd"]  += h["gain_loss_usd"]
    TARGETS = {"US Equity":45, "Intl Equity":15, "Fixed Income":25, "Real Estate":5, "Cash":5}
    rebal = [{"asset_class":ac, "current_pct":round(class_map.get(ac,{}).get("allocation_pct",0),2),
              "target_pct":t, "diff":round(class_map.get(ac,{}).get("allocation_pct",0)-t,2),
              "action":"REDUCE" if class_map.get(ac,{}).get("allocation_pct",0)>t else "INCREASE"}
             for ac, t in TARGETS.items() if abs(class_map.get(ac,{}).get("allocation_pct",0)-t)>=3]
    ranked = sorted([h for h in holdings if h.get("gain_loss_pct") is not None],
                    key=lambda x: x["gain_loss_pct"], reverse=True)
    spy_pcts = [h["gain_loss_pct"] for h in SAMPLE_PORTFOLIOS if h["ticker"]=="SPY"]
    spy_ret  = sum(spy_pcts)/len(spy_pcts) if spy_pcts else 10.0
    return {
        "total_aum":round(total_usd,2), "total_gain_loss_usd":round(total_gain,2),
        "total_return_pct":round(total_ret,2), "volatility_proxy":round(vol,2),
        "vol_level":"HIGH" if vol>30 else "MODERATE" if vol>15 else "LOW",
        "asset_class_breakdown":class_map, "top_performers":ranked[:5],
        "bottom_performers":ranked[-5:], "rebalancing_signals":rebal,
        "spy_return_pct":round(spy_ret,2), "alpha_vs_spy":round(total_ret-spy_ret,2),
        "gl_range":{"min":round(min(gl_pcts),2) if gl_pcts else 0,
                    "max":round(max(gl_pcts),2) if gl_pcts else 0},
    }


def save_portfolio_excel(portfolio: dict, path: Path) -> None:
    """Write portfolio holdings to a formatted 4-sheet Excel file."""
    holdings = portfolio.get("holdings", [])
    rows = [{"Type of Investment":h.get("asset_class",""), "Ticker":h["ticker"],
             "Investment Type":h.get("investment_type",""),
             "% Allocation":round(h.get("allocation_pct",0),2),
             "Amount ($)":round(h.get("amount_usd",0),2),
             "Company / Name":h.get("company_name",""),
             "Gain/Loss ($)":round(h.get("gain_loss_usd",0) if h.get("gain_loss_usd") else 0,2),
             "Source Account(s)":", ".join(h.get("source_accounts",[]) if h.get("source_accounts") else [h.get("source_account","")])} for h in holdings]
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        pd.DataFrame([{"Portfolio Name":portfolio.get("name",""),
                       "Description":portfolio.get("description",""),
                       "Total Holdings":len(holdings),
                       "Total Allocation %":round(sum(h.get("allocation_pct",0) for h in holdings),2),
                       "Generated":datetime.now().strftime("%Y-%m-%d %H:%M")
                      }]).to_excel(writer, sheet_name="Summary", index=False)
        df.to_excel(writer, sheet_name="Holdings", index=False)
        from openpyxl.styles import Font, PatternFill, Alignment
        for ws in writer.sheets.values():
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(
                    max(len(str(c.value or "")) for c in col)+4, 55)
            for cell in ws[1]:
                cell.fill = PatternFill("solid", fgColor="1F3864")
                cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
                cell.alignment = Alignment(horizontal="center")


# Pre-compute consolidated analytics
CONSOLIDATED = consolidate_holdings(SAMPLE_PORTFOLIOS)
ANALYTICS    = compute_analytics(CONSOLIDATED)
PORTFOLIO_CONTEXT = (
    f"Consolidated portfolio ({len(CONSOLIDATED)} holdings, ${TOTAL_AUM:,.0f} AUM, 5 accounts):\n" +
    ", ".join(f"{h['ticker']} ({h.get('investment_type','?')}, {h['allocation_pct']:.1f}%)" for h in CONSOLIDATED)
)

print(f"✅ Analytics ready: {len(CONSOLIDATED)} holdings, return={ANALYTICS['total_return_pct']:.1f}%, "
      f"vol={ANALYTICS['volatility_proxy']:.1f} ({ANALYTICS['vol_level']})")

## 6. Persistent Investor Profile & Shared Session State

This is the core new feature. `INVESTOR_PROFILE` accumulates information
across turns. `extract_profile_updates()` uses the LLM to parse each message
for any new profile fields, which are then merged in. Every Agent 1 call
receives the **complete, up-to-date profile** as part of its prompt.

In [ ]:
# ── Investor Profile — accumulated across all turns ───────────────────────────
INVESTOR_PROFILE: dict = {
    "age"              : None,   # e.g. 45
    "risk_tolerance"   : None,   # conservative | moderate | aggressive
    "time_horizon_yrs" : None,   # e.g. 15
    "primary_goal"     : None,   # retirement | growth | income | education | purchase
    "experience_level" : None,   # beginner | intermediate | advanced
    "preferred_instruments": [],  # ["ETF", "stocks", "bonds", ...]
    "excluded_sectors" : [],      # ["real estate", "crypto", ...]
    "sector_preferences": [],     # ["tech", "healthcare", ...]
    "investment_amount": None,   # total investable $ if mentioned
    "tax_situation"    : None,   # taxable | IRA | Roth | 401k
    "liquidity_needs"  : None,   # high | medium | low
    "last_updated"     : None,
}

# ── Session State ─────────────────────────────────────────────────────────────
SESSION_STATE: dict = {
    "draft_portfolio"  : None,   # dict — pending user approval
    "draft_holdings_text": None, # LLM-generated holdings table string
    "saved_portfolio"  : None,   # dict — after confirm_portfolio(approved=True)
    "portfolio_status" : "none", # none | draft | approved | rejected
    "dispatch_log"     : [],     # [{query, tool, elapsed_s}]
    "turn_count"       : 0,
}


def profile_as_string() -> str:
    """Return the investor profile as a readable string for injection into prompts."""
    p = INVESTOR_PROFILE
    filled = {k: v for k, v in p.items()
              if v is not None and v != [] and k != "last_updated"}
    if not filled:
        return "No investor profile collected yet."
    lines = []
    for k, v in filled.items():
        label = k.replace("_", " ").title()
        val   = ", ".join(v) if isinstance(v, list) else str(v)
        lines.append(f"  • {label}: {val}")
    return "Accumulated Investor Profile:\n" + "\n".join(lines)


def extract_and_merge_profile(user_message: str) -> dict:
    """
    Use the LLM to extract any investor profile fields from the user's message
    and merge them into the global INVESTOR_PROFILE.
    Returns a dict of the newly extracted fields (may be empty).
    """
    extractor_llm = load_llm_from_env(temperature=0)
    system = """\
Extract investor profile information from the user's message.
Return ONLY a valid JSON object with any of these keys that are mentioned
(omit keys not mentioned — do not guess or infer):
{
  "age": <integer>,
  "risk_tolerance": "conservative" | "moderate" | "aggressive",
  "time_horizon_yrs": <integer>,
  "primary_goal": "retirement" | "growth" | "income" | "education" | "purchase",
  "experience_level": "beginner" | "intermediate" | "advanced",
  "preferred_instruments": ["ETF", "stocks", "bonds", "crypto", ...],
  "excluded_sectors": ["real estate", "crypto", ...],
  "sector_preferences": ["tech", "healthcare", "energy", ...],
  "investment_amount": <float>,
  "tax_situation": "taxable" | "IRA" | "Roth" | "401k",
  "liquidity_needs": "high" | "medium" | "low"
}
Return {} if nothing relevant is mentioned. No markdown, no explanation."""

    try:
        resp = extractor_llm.invoke([
            SystemMessage(content=system),
            HumanMessage(content=user_message)
        ])
        raw = resp.content.strip().lstrip("```json").rstrip("```").strip()
        updates = json.loads(raw)
    except Exception:
        return {}

    # Merge into global profile
    for k, v in updates.items():
        if k in INVESTOR_PROFILE:
            if isinstance(INVESTOR_PROFILE[k], list) and isinstance(v, list):
                # Merge lists (avoid duplicates)
                for item in v:
                    if item not in INVESTOR_PROFILE[k]:
                        INVESTOR_PROFILE[k].append(item)
            else:
                INVESTOR_PROFILE[k] = v
    if updates:
        INVESTOR_PROFILE["last_updated"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return updates


print("✅ Investor profile and session state initialised")
print("   Profile fields:", list(INVESTOR_PROFILE.keys()))

## 7. Define All Orchestrator Tools

Five tools — each is a focused specialist agent or utility:
1. `invoke_portfolio_designer` — builds a portfolio using accumulated profile
2. `confirm_portfolio` — NEW: approve draft → save → trigger consolidation
3. `invoke_portfolio_analyst` — analytics on the 5-account consolidated portfolio
4. `invoke_investment_educator` — plain-English concept explanations
5. `search_web` — live market data via Serper

In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the web for current market data, ETF prices, or investment news.
    Use for: current prices, recent market events, live benchmark returns.
    Do NOT use for basic concept definitions.
    """
    try:
        return GoogleSerperAPIWrapper().run(query)
    except Exception as e:
        return f"Search unavailable: {e}"


print("search_web defined")

In [ ]:
@tool
def invoke_portfolio_designer(additional_context: str = "") -> str:
    """
    Invoke Agent 1 (Portfolio Designer) to build or refine a portfolio.

    This tool automatically injects the full accumulated investor profile
    from the session state, so it does NOT need to ask for information that
    has already been provided in earlier turns.

    Use this tool when the user:
    - Wants to build a new portfolio ('help me build a portfolio')
    - Provides profile details ('I am 45, moderate risk, retiring in 15 years')
    - Wants to refine a draft portfolio ('change bonds to 20%', 'add healthcare')
    - Asks for an allocation recommendation based on their profile

    Args:
        additional_context: Any extra context from the user's current message
                            beyond what is already in the investor profile.

    Returns:
        A complete portfolio recommendation table with tickers, allocations,
        and rationale. The portfolio is saved as a DRAFT pending confirmation.
    """
    profile_str = profile_as_string()

    # Check if there is an existing draft to refine
    existing_draft = ""
    if SESSION_STATE["portfolio_status"] == "draft" and SESSION_STATE["draft_holdings_text"]:
        existing_draft = (
            f"\n\nEXISTING DRAFT (user wants to refine this):\n"
            f"{SESSION_STATE['draft_holdings_text'][:1200]}"
        )

    system = f"""\
You are an expert portfolio designer. Design a specific, complete portfolio
using the investor's accumulated profile below. Follow these rules:

1. USE the profile immediately — do NOT ask for information already provided.
2. Provide 8-10 holdings with specific tickers.
3. Allocations MUST sum to exactly 100%.
4. Present as a clear markdown table:
   | Ticker | Name | Type | % Allocation | Rationale |
5. After the table, add a 2-sentence summary of the strategy.
6. End with: '💼 Type "approve" or "yes, save it" to save this portfolio,
   or describe any changes you'd like.'
7. If risk/horizon info is missing, use reasonable defaults for a
   balanced moderate-risk 10-year investor and state your assumptions.
8. Educational purposes only — not financial advice.

{profile_str}
{existing_draft}
"""
    designer_llm = load_llm_from_env(temperature=0.1)
    request = additional_context or "Please design my portfolio based on my profile."
    response = designer_llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=request)
    ])

    # Store as draft — requires confirmation before saving
    SESSION_STATE["draft_holdings_text"] = response.content
    SESSION_STATE["draft_portfolio"]     = {
        "name"           : f"Designed Portfolio — {datetime.now().strftime('%Y-%m-%d')}",
        "description"    : profile_str[:300],
        "design_response": response.content,
        "profile_snapshot": copy.deepcopy(INVESTOR_PROFILE),
    }
    SESSION_STATE["portfolio_status"] = "draft"

    return response.content


print("invoke_portfolio_designer defined")

In [ ]:
@tool
def confirm_portfolio(approved: bool, change_request: str = "") -> str:
    """
    Confirm, reject, or request changes to the pending draft portfolio.

    This is the HUMAN-IN-THE-LOOP approval gate. Call this tool when the user:
    - Approves  : 'yes', 'save it', 'looks good', 'approve', 'confirm'
    - Rejects   : 'no', 'cancel', 'start over', 'discard'
    - Requests a change while also approving: 'save it but change bonds to 20%'
      → set approved=False, change_request='change bonds to 20%'
      → Orchestrator will call invoke_portfolio_designer again with the change,
        then call confirm_portfolio again after user re-reviews.

    When approved=True:
      - Saves portfolio to ../data/outputs/portfolio.json + portfolio.xlsx
      - Automatically merges with the 5 existing accounts
      - Saves consolidated view to ../data/outputs/user1/consolidated_portfolio.json

    Args:
        approved     : True = save and consolidate. False = reject or change needed.
        change_request: (optional) Description of changes if not approved.

    Returns:
        Confirmation message with saved file paths, or rejection/change message.
    """
    if SESSION_STATE["portfolio_status"] != "draft":
        return ("⚠️ No draft portfolio is pending. "
                "Please ask me to build a portfolio first.")

    draft = SESSION_STATE["draft_portfolio"]

    if not approved:
        if change_request:
            SESSION_STATE["portfolio_status"] = "draft"  # keep draft, allow refinement
            return (
                f"📝 Got it — I'll revise the portfolio with your change:\n"
                f'   "{change_request}"\n'
                f"The Orchestrator will regenerate the portfolio. "
                f"Review it and say 'approve' when ready."
            )
        else:
            SESSION_STATE["portfolio_status"] = "rejected"
            SESSION_STATE["draft_portfolio"]  = None
            SESSION_STATE["draft_holdings_text"] = None
            return "❌ Portfolio discarded. Start fresh anytime by asking me to build a new one."

    # ── APPROVED: Save + Consolidate ─────────────────────────────────────────

    # Parse holdings from the LLM-generated markdown table
    # We store the text response and a minimal structured form
    holdings_raw = _parse_holdings_from_text(draft.get("design_response", ""))

    portfolio_obj = {
        "name"        : draft["name"],
        "description" : draft["description"],
        "holdings"    : holdings_raw,
        "approved_at" : datetime.now().isoformat(),
        "investor_profile": draft.get("profile_snapshot", {}),
    }

    # Save Agent 1 JSON
    AGENT1_JSON.parent.mkdir(parents=True, exist_ok=True)
    with open(AGENT1_JSON, "w") as f:
        json.dump(portfolio_obj, f, indent=2)

    # Save Agent 1 Excel
    save_portfolio_excel(portfolio_obj, AGENT1_EXCEL)

    # Auto-consolidate: merge new portfolio with existing 5 accounts
    merged_raw = SAMPLE_PORTFOLIOS + [
        {"ticker": h.get("ticker","").upper(),
         "company_name": h.get("name", h.get("company_name","")),
         "investment_type": h.get("investment_type","ETF"),
         "asset_class": h.get("asset_class","US Equity"),
         "amount_usd": float(h.get("amount_usd", 0) or 0),
         "gain_loss_usd": 0.0, "gain_loss_pct": None,
         "source_file": "New Portfolio (Designed)",
         "source_account": "New Portfolio",
         "shares": 0.0}
        for h in holdings_raw if h.get("ticker")
    ]
    consolidated_holdings = consolidate_holdings(merged_raw)
    consolidated_portfolio = {
        "name"        : "Full Consolidated Portfolio (incl. new design)",
        "description" : f"5 existing accounts + new designed portfolio ({draft['name']})",
        "source_files": [m["source_file"] for m in ACCOUNT_METADATA] + ["New Portfolio (Designed)"],
        "holdings"    : consolidated_holdings,
        "generated_at": datetime.now().isoformat(),
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(CONSOLIDATED_JSON, "w") as f:
        json.dump(consolidated_portfolio, f, indent=2)
    save_portfolio_excel(consolidated_portfolio, CONSOLIDATED_EXCEL)

    SESSION_STATE["portfolio_status"] = "approved"
    SESSION_STATE["saved_portfolio"]  = portfolio_obj

    total_alloc = round(sum(h.get("allocation_pct",0) for h in holdings_raw), 1)
    return (
        f"✅ Portfolio APPROVED and saved!\n\n"
        f"   📄 Agent 1 JSON  → {AGENT1_JSON}\n"
        f"   📊 Agent 1 Excel → {AGENT1_EXCEL}\n"
        f"   🔀 Auto-consolidated with 5 existing accounts:\n"
        f"      📄 JSON  → {CONSOLIDATED_JSON}\n"
        f"      📊 Excel → {CONSOLIDATED_EXCEL}\n\n"
        f"   Holdings in new portfolio : {len(holdings_raw)} (allocation total: {total_alloc}%)\n"
        f"   Holdings after full merge : {len(consolidated_holdings)} unique tickers\n\n"
        f"You can now ask 'How volatile is my portfolio?' to analyse the "
        f"updated consolidated view."
    )


def _parse_holdings_from_text(text: str) -> list[dict]:
    """
    Use the LLM to extract structured holdings from the designer's
    markdown table response. Returns a list of holding dicts.
    """
    extractor = load_llm_from_env(temperature=0)
    system = """\
Extract the portfolio holdings from the text below.
Return ONLY a valid JSON array — no markdown, no explanation.
Each element: {"ticker":"VTI","name":"Vanguard Total Stock Market ETF",
               "investment_type":"ETF","asset_class":"US Equity",
               "allocation_pct":25.0,"amount_usd":null,"rationale":"brief"}
If allocation_pct is missing, distribute equally. Sum must be 100."""
    try:
        resp = extractor.invoke([
            SystemMessage(content=system),
            HumanMessage(content=text[:3000])
        ])
        raw = resp.content.strip().lstrip("```json").rstrip("```").strip()
        holdings = json.loads(raw)
        # Normalise allocations to 100%
        total = sum(h.get("allocation_pct",0) for h in holdings)
        if total > 0:
            for h in holdings:
                h["allocation_pct"] = round(h.get("allocation_pct",0)/total*100, 2)
        return holdings if isinstance(holdings, list) else []
    except Exception:
        return []


print("confirm_portfolio defined  ✅  (human-in-the-loop approval gate)")

In [ ]:
@tool
def invoke_portfolio_analyst(user_request: str) -> str:
    """
    Invoke Agent 2 (Portfolio Analyst) to answer quantitative questions
    about the consolidated portfolio.

    Use for: performance, volatility, benchmark comparison, rebalancing signals,
    top/bottom performers, asset allocation breakdown.

    Args:
        user_request: The analytics question (e.g. 'How volatile is my portfolio?')

    Returns:
        Specific numbers + plain-English explanation + actionable insight.
    """
    a = ANALYTICS
    c = CONSOLIDATED
    asset_str  = "  |  ".join(f"{ac}: {round(v['allocation_pct'],1)}%" for ac,v in a["asset_class_breakdown"].items())
    top3_str   = ", ".join(f"{h['ticker']} (+{h['gain_loss_pct']:.1f}%)" for h in a["top_performers"][:3])
    bot3_str   = ", ".join(f"{h['ticker']} ({h['gain_loss_pct']:+.1f}%)" for h in a["bottom_performers"][:3])
    rebal_str  = "  |  ".join(
        f"{r['asset_class']}: {r['current_pct']:.1f}% vs target {r['target_pct']}% → {r['action']}"
        for r in a["rebalancing_signals"]
    ) or "No significant drift detected."
    data = (
        f"PORTFOLIO DATA ({len(c)} holdings, 5 accounts):\n"
        f"• AUM: ${a['total_aum']:,.0f}  |  Total G/L: ${a['total_gain_loss_usd']:+,.0f}\n"
        f"• Return: {a['total_return_pct']:+.1f}%  |  S&P 500: {a['spy_return_pct']:+.1f}%  |  Alpha: {a['alpha_vs_spy']:+.1f}%\n"
        f"• Volatility Proxy: {a['volatility_proxy']:.1f} ({a['vol_level']})  |  Range: {a['gl_range']['min']:+.1f}% to {a['gl_range']['max']:+.1f}%\n"
        f"• Allocation: {asset_str}\n"
        f"• Top 3: {top3_str}\n"
        f"• Bottom 3: {bot3_str}\n"
        f"• Rebalancing: {rebal_str}"
    )
    system = (
        "You are an expert portfolio analyst. Answer using ONLY the data below. "
        "Be specific with every number. Explain what the metrics mean in plain "
        "business language. End with one concrete actionable recommendation. "
        "Close with: 'Educational purposes only — not financial advice.'\n\n" + data
    )
    resp = load_llm_from_env().invoke([
        SystemMessage(content=system),
        HumanMessage(content=user_request)
    ])
    SESSION_STATE["dispatch_log"][-1]["agent"] = "Agent2" if SESSION_STATE["dispatch_log"] else None
    return resp.content


print("invoke_portfolio_analyst defined")

In [ ]:
@tool
def invoke_investment_educator(concept: str) -> str:
    """
    Invoke Agent 3 (Investment Educator) to explain an investment concept
    in plain language, personalised to the user's actual holdings.

    Use for: 'What is X?', 'Explain X', 'How does X work?', 'Define X'
    where X is any investment/finance term (beta, DCA, ETF, alpha, etc.)

    Args:
        concept: The concept or question to explain.

    Returns:
        A 200-300 word explanation with definition, analogy, portfolio
        application (using real tickers), and practical takeaway.
    """
    system = (
        "You are a warm, expert investment educator. Explain every concept using:\n"
        "1. Simple one-sentence definition\n"
        "2. A relatable non-financial analogy\n"
        "3. How it applies to the user's specific holdings (use REAL tickers below!)\n"
        "4. One practical takeaway\n\n"
        f"{PORTFOLIO_CONTEXT}\n\n"
        "Keep to 200-300 words. End with:\n"
        "'📚 Remember: this is for educational purposes only, not financial advice.'\n"
        "Then invite a follow-up question."
    )
    resp = load_llm_from_env(temperature=0.3).invoke([
        SystemMessage(content=system),
        HumanMessage(content=concept)
    ])
    return resp.content


ORCHESTRATOR_TOOLS = [
    invoke_portfolio_designer,
    confirm_portfolio,
    invoke_portfolio_analyst,
    invoke_investment_educator,
    search_web,
]

print(f"invoke_investment_educator defined")
print(f"\n✅ All {len(ORCHESTRATOR_TOOLS)} Orchestrator tools ready:")
for t in ORCHESTRATOR_TOOLS:
    print(f"   • {t.name}")

## 8. Orchestrator System Prompt

In [ ]:
ORCHESTRATOR_SYSTEM = f"""\
You are the Master Orchestrator of a professional investment management system.
You have 5 specialist tools. Pick EXACTLY ONE tool per user message.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOOL SELECTION RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

① invoke_portfolio_designer
   → User wants to BUILD or REFINE a portfolio
   → User provides profile info: age, risk, timeline, goals
   → Examples: 'help me build', 'I am 45, moderate risk', 'change bonds to 20%'
   → The tool auto-injects the full accumulated investor profile

② confirm_portfolio
   → User APPROVES the draft: 'yes', 'save it', 'looks good', 'approve', 'confirm'
   → User REJECTS: 'no', 'cancel', 'discard', 'start over'
   → Call with approved=True if any approval language is detected
   → Call with approved=False + change_request if user wants changes
   → ONLY call if portfolio_status = 'draft' (a portfolio has been designed)

③ invoke_portfolio_analyst
   → User asks QUANTITATIVE questions about the existing portfolio
   → Examples: 'how volatile', 'compare to S&P', 'rebalance', 'top performers'

④ invoke_investment_educator
   → User wants to LEARN a concept
   → Examples: 'what is beta', 'explain DCA', 'what is an ETF'

⑤ search_web
   → User needs CURRENT data: live prices, recent news, today's market data
   → Example: 'what is VTI trading at today?'

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PORTFOLIO WORKFLOW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. User provides profile info → invoke_portfolio_designer (accumulates in state)
2. User asks to build → invoke_portfolio_designer → draft created
3. User reviews draft
4a. User approves → confirm_portfolio(approved=True) → saved + consolidated
4b. User wants changes → invoke_portfolio_designer (refines using change request)
4c. User cancels → confirm_portfolio(approved=False)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONTEXT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Portfolio status : {{portfolio_status}}
Existing portfolio: {len(CONSOLIDATED)} consolidated holdings, ${TOTAL_AUM:,.0f} AUM
Top holdings     : {', '.join(h['ticker'] for h in CONSOLIDATED[:6])}

IMPORTANT: Always call a tool. Never answer from your own knowledge.
"""

print(f"Orchestrator system prompt: {len(ORCHESTRATOR_SYSTEM):,} chars")

## 9. Build the Orchestrator Graph

In [ ]:
orch_llm            = load_llm_from_env()
orch_llm_with_tools = orch_llm.bind_tools(ORCHESTRATOR_TOOLS)


def orchestrator_node(state: MessagesState):
    """The Orchestrator LLM — decides which specialist tool to call."""
    # Inject live portfolio status into the system prompt
    live_prompt = ORCHESTRATOR_SYSTEM.replace(
        "{portfolio_status}", SESSION_STATE["portfolio_status"]
    )
    messages = [SystemMessage(content=live_prompt)] + state["messages"]
    return {"messages": [orch_llm_with_tools.invoke(messages)]}


def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    return "tools" if hasattr(last, "tool_calls") and last.tool_calls else "__end__"


graph_builder = StateGraph(MessagesState)
graph_builder.add_node("orchestrator", orchestrator_node)
graph_builder.add_node("tools", ToolNode(ORCHESTRATOR_TOOLS))
graph_builder.add_edge(START, "orchestrator")
graph_builder.add_conditional_edges("orchestrator", should_continue, ["tools", "__end__"])
graph_builder.add_edge("tools", "orchestrator")

memory     = MemorySaver()   # single shared memory across ALL session turns
orch_graph = graph_builder.compile(checkpointer=memory)

print("✅ Orchestrator Graph compiled")
print(f"   Tools : {[t.name for t in ORCHESTRATOR_TOOLS]}")
print(f"   Memory: single MemorySaver (shared across turns)")

In [ ]:
try:
    display(Image(orch_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Diagram skipped — install pygraphviz for visualisation)")

## 10. Chat Helper

In [ ]:
TOOL_LABELS = {
    "invoke_portfolio_designer" : "🏗️  Agent 1 — Portfolio Designer",
    "confirm_portfolio"         : "✅  confirm_portfolio",
    "invoke_portfolio_analyst"  : "📊  Agent 2 — Portfolio Analyst",
    "invoke_investment_educator": "📚  Agent 3 — Investment Educator",
    "search_web"                : "🔍  search_web",
}


def chat(user_input: str, session_id: str = "final", verbose: bool = True) -> str:
    """
    Send a message to the Orchestrator.

    Before forwarding, profile information is extracted from the message
    and merged into INVESTOR_PROFILE automatically.

    Args:
        user_input : The user's message
        session_id : Thread ID — all turns with same ID share conversation memory
        verbose    : Print which tool was dispatched

    Returns:
        The specialist agent's response as a string
    """
    SESSION_STATE["turn_count"] += 1
    t0 = time.time()

    # Step 1: Extract and merge profile info before routing
    new_fields = extract_and_merge_profile(user_input)
    if new_fields and verbose:
        print(f"  📋 Profile updated: {list(new_fields.keys())}")

    # Step 2: Run the Orchestrator graph
    config = {"configurable": {"thread_id": session_id}}
    result = None
    for event in orch_graph.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
        stream_mode="values",
    ):
        result = event

    elapsed  = round(time.time() - t0, 2)
    response = result["messages"][-1].content if result else "(no response)"

    # Step 3: Log dispatch
    tool_called = "unknown"
    for msg in result.get("messages", []):
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tool_called = msg.tool_calls[0]["name"]
            break

    label = TOOL_LABELS.get(tool_called, tool_called)
    SESSION_STATE["dispatch_log"].append({
        "turn": SESSION_STATE["turn_count"],
        "query": user_input[:60],
        "tool": tool_called,
        "label": label,
        "elapsed_s": elapsed,
        "profile_fields_added": list(new_fields.keys()) if new_fields else [],
    })

    if verbose:
        print(f"  🎯 {label}  [{elapsed}s]")

    return response


DIVIDER = "=" * 72
print("✅ System ready! Usage: chat('your message')")

---
## 11. Part A — The 5 Standard Test Queries

Same 5 queries as Notebooks 14 & 15 — all share `session_id='final_test'`
so the Orchestrator has full context across turns.

In [ ]:
SESSION = "final_test"

print(DIVIDER)
print("QUERY 1: Help me build a long-term portfolio.")
print(DIVIDER)
q = "Help me build a long-term portfolio."
print(f"User: {q}\n")
print(chat(q, SESSION))

In [ ]:
print(DIVIDER)
print("QUERY 2: I am 45, moderate risk, retiring in 15 years.")
print(DIVIDER)
q = "I am 45, moderate risk, retiring in 15 years — what allocation should I use?"
print(f"User: {q}\n")
print(chat(q, SESSION))

In [ ]:
print(DIVIDER)
print("QUERY 3: What is beta?")
print(DIVIDER)
q = "What is beta?"
print(f"User: {q}\n")
print(chat(q, SESSION))

In [ ]:
print(DIVIDER)
print("QUERY 4: What is dollar-cost averaging?")
print(DIVIDER)
q = "What is dollar-cost averaging?"
print(f"User: {q}\n")
print(chat(q, SESSION))

In [ ]:
print(DIVIDER)
print("QUERY 5: How volatile is my portfolio?")
print(DIVIDER)
q = "How volatile is my portfolio?"
print(f"User: {q}\n")
print(chat(q, SESSION))

---
## 12. Part B — Full Portfolio Design → Confirm → Save → Consolidate Flow

This section demonstrates the complete multi-turn workflow, showing how
the `confirm_portfolio` tool and persistent investor profile work together.

In [ ]:
print(DIVIDER)
print("CONFIRM FLOW — Turn 1: Provide profile (age + risk)")
print(DIVIDER)
q = "I'm 38 years old with an aggressive risk tolerance."
print(f"User: {q}\n")
print(chat(q, session_id="confirm_flow"))
print(f"\n📋 Profile so far: {profile_as_string()}")

In [ ]:
print(DIVIDER)
print("CONFIRM FLOW — Turn 2: Add more profile details")
print(DIVIDER)
q = "My goal is retirement in 25 years. I prefer ETFs and tech stocks. No real estate."
print(f"User: {q}\n")
print(chat(q, session_id="confirm_flow"))
print(f"\n📋 Profile so far: {profile_as_string()}")

In [ ]:
print(DIVIDER)
print("CONFIRM FLOW — Turn 3: Request portfolio build")
print(DIVIDER)
q = "Great, now build me a portfolio based on everything I've told you."
print(f"User: {q}\n")
print(chat(q, session_id="confirm_flow"))
print(f"\n📋 Portfolio status: {SESSION_STATE['portfolio_status']}")

In [ ]:
print(DIVIDER)
print("CONFIRM FLOW — Turn 4: Request a change before saving")
print(DIVIDER)
q = "I like it, but can you reduce the bond allocation and put more into international ETFs?"
print(f"User: {q}\n")
print(chat(q, session_id="confirm_flow"))

In [ ]:
print(DIVIDER)
print("CONFIRM FLOW — Turn 5: Approve and save")
print(DIVIDER)
q = "Perfect, that looks great. Yes, please save it."
print(f"User: {q}\n")
print(chat(q, session_id="confirm_flow"))

---
## 13. Verify Saved Output

In [ ]:
print(DIVIDER)
print("SAVED FILES VERIFICATION")
print(DIVIDER)

for path, label in [
    (AGENT1_JSON,        "Agent 1 Portfolio (designed)"),
    (AGENT1_EXCEL,       "Agent 1 Excel"),
    (CONSOLIDATED_JSON,  "Consolidated Portfolio"),
    (CONSOLIDATED_EXCEL, "Consolidated Excel"),
]:
    status = "✅" if path.exists() else "⚠️  not yet created"
    size   = f"({path.stat().st_size:,} bytes)" if path.exists() else ""
    print(f"  {status}  {label:<30} {path.name}  {size}")

# Show consolidated holdings table if available
if CONSOLIDATED_JSON.exists():
    with open(CONSOLIDATED_JSON) as f:
        saved = json.load(f)
    h_list = saved.get("holdings", [])
    total  = sum(h.get("allocation_pct",0) for h in h_list)
    print(f"\n  Portfolio : {saved.get('name','')}")
    print(f"  Holdings  : {len(h_list)}   |   Total alloc: {total:.2f}%")
    print(f"\n  {'Ticker':<8} {'Alloc%':>7}  {'Amount':>12}  {'Type':<15} {'Name'}")
    print("  " + "-"*80)
    for h in h_list[:15]:   # show top 15
        amt = f"${h.get('amount_usd',0):>10,.0f}" if h.get('amount_usd') else "           —"
        print(f"  {h['ticker']:<8} {h.get('allocation_pct',0):>6.2f}%  {amt}  "
              f"{str(h.get('investment_type','')):<15} {str(h.get('company_name',''))[:38]}")
    if len(h_list) > 15:
        print(f"  ... and {len(h_list)-15} more holdings")
    print("  " + "-"*80)
    print(f"  {'TOTAL':<8} {total:>6.2f}%")

---
## 14. Full Session Dispatch Log

In [ ]:
print(DIVIDER)
print("FULL SESSION DISPATCH LOG")
print(DIVIDER)
print(f"  {'Turn':<5} {'Query':<55} {'Tool Dispatched':<35} {'Time':>6} {'Profile Updates'}")
print("  " + "-" * 115)
for e in SESSION_STATE["dispatch_log"]:
    q_s    = e['query'][:52] + ".." if len(e['query']) > 52 else e['query']
    label  = TOOL_LABELS.get(e['tool'], e['tool']).replace("🏗️","").replace("📊","").replace("📚","").replace("✅","").replace("🔍","").strip()
    p_upd  = ", ".join(e.get('profile_fields_added', [])) or ""
    print(f"  {e['turn']:<5} {q_s:<55} {label:<35} {e['elapsed_s']:>5.1f}s  {p_upd}")
print("  " + "-" * 115)
print(f"\n  Total turns: {SESSION_STATE['turn_count']}")
print(f"  Final investor profile:")
print("\n".join(f"    {l}" for l in profile_as_string().split("\n")))

---
## 15. Interactive Mode

In [ ]:
print(DIVIDER)
print("FINAL CONSOLIDATED INVESTMENT SYSTEM — Interactive Mode")
print(DIVIDER)
print("All 5 tools active. Try a full workflow:")
print("  1. 'I am 30, aggressive risk, 30-year horizon, prefer ETFs'")
print("  2. 'Build me a growth-focused portfolio'")
print("  3. 'Change the bond allocation to 10%'")
print("  4. 'Looks great, save it'")
print("  5. 'How volatile is my consolidated portfolio?'")
print("  6. 'What is the Sharpe ratio?'")
print("Type 'profile' to see accumulated investor profile.")
print("Type 'log' to see dispatch history. Type 'quit' to stop.")
print(DIVIDER)

INT_SESSION = "interactive_final"
while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nSession ended. Thank you for using the Investment System.")
        break
    if user_input.lower() == "profile":
        print(f"\n{profile_as_string()}")
        print(f"Portfolio status: {SESSION_STATE['portfolio_status']}")
        continue
    if user_input.lower() == "log":
        for e in SESSION_STATE["dispatch_log"]:
            print(f"  Turn {e['turn']}: {TOOL_LABELS.get(e['tool'],e['tool'])} | {e['query'][:50]}")
        continue
    response = chat(user_input, session_id=INT_SESSION)
    print(f"\nAgent: {response}")

---
## 16. System Summary

### What's New vs Notebook 15 (Previous Orchestrator)

| Feature | NB 15 Orchestrator | NB 16 Final (This) |
|---------|-------------------|--------------------|
| `confirm_portfolio` tool | ❌ Missing | ✅ Full approve/reject/change flow |
| Persistent investor profile | ❌ Single-turn | ✅ Accumulates across all turns |
| Profile extraction | ❌ None | ✅ LLM extracts fields each turn |
| Profile injected into Agent 1 | ❌ No | ✅ Full profile in every call |
| Post-approval auto-consolidation | ❌ No | ✅ Merges with 5 existing accounts |
| Draft state management | ❌ No | ✅ none/draft/approved/rejected |
| Refinement loop | ❌ No | ✅ change → redesign → re-confirm |
| Excel + JSON on save | ❌ JSON only | ✅ Both written on approval |

### Tool Dispatch Summary

| Tool | Trigger | Agent Behind It |
|------|---------|----------------|
| `invoke_portfolio_designer` | Build/refine requests | Agent 1 (with full profile) |
| `confirm_portfolio` | approve/reject/change | Human-in-the-loop gate |
| `invoke_portfolio_analyst` | Quantitative questions | Agent 2 (full analytics) |
| `invoke_investment_educator` | "What is X?" | Agent 3 (personalised) |
| `search_web` | Current prices/news | Serper API |

### Portfolio State Machine
```
none ──► draft ──► approved ──► (consolidated & saved)
          │
          ├──► rejected  (user says cancel)
          └──► draft     (user requests changes → redesign)
```

### Investor Profile Accumulation
```
Each turn → extract_and_merge_profile(message)
                     │
           Updates INVESTOR_PROFILE dict
                     │
           invoke_portfolio_designer reads
           the FULL profile via profile_as_string()
```

### Output Files
```
../data/outputs/portfolio.json            ← Agent 1 designed portfolio
../data/outputs/portfolio.xlsx            ← Agent 1 Excel (2 sheets)
../data/outputs/user1/consolidated_portfolio.json   ← All accounts merged
../data/outputs/user1/consolidated_portfolio.xlsx   ← Consolidated Excel
```